# Lab 1 - Data Collection and Pre-Processing

This notebook demonstrates an end-to-end data engineering workflow using an e-commerce sales dataset. The project follows a 12-step process covering data ingestion, Python data structures, cleaning, transformation, feature engineering, aggregation, and serialization.

The primary dataset is based on the public **1000 Sales Records** sample from ExcelBIAnalytics. The first 500 records will be used for this lab and enriched with additional e-commerce fields required for the analysis.

## Step 1 - Hello, Data!

The raw sales dataset is loaded from the `data` folder using Pandas. The first 500 records are selected to meet the requirements of this lab. The first three rows are displayed to verify that the data was loaded successfully.

In [1]:
# Import Pandas for working with tabular data
import pandas as pd

# Load the raw sales CSV and keep the first 500 records
df = pd.read_csv("data/1000 Sales Records.csv").head(500)

# Display the first three rows
df.head(3)

,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Middle East and North Africa,Libya,Cosmetics,Offline,M,10/18/2014,686800706,10/31/2014,8446,437.20,263.33,3692591.20,2224085.18,1468506.02
1,North America,Canada,Vegetables,Online,M,11/7/2011,185941302,12/8/2011,3018,154.06,90.93,464953.08,274426.74,190526.34
2,Middle East and North Africa,Libya,Baby Food,Offline,C,10/31/2016,246222341,12/9/2016,1517,255.28,159.42,387259.76,241840.14,145419.62


In [2]:
# Display the column names in the raw dataset
print(df.columns.tolist())

['Region', 'Country', 'Item Type', 'Sales Channel', 'Order Priority', 'Order Date', 'Order ID', 'Ship Date', 'Units Sold', 'Unit Price', 'Unit Cost', 'Total Revenue', 'Total Cost', 'Total Profit']


In [3]:
# Create a working copy so the original imported data remains unchanged
sales_df = df.copy()

# Rename existing columns to match the e-commerce fields used in this lab
sales_df = sales_df.rename(columns={
    "Order Date": "date",
    "Item Type": "product",
    "Unit Price": "price",
    "Units Sold": "quantity"
})

# Display the first three rows of the renamed data
sales_df[["date", "product", "price", "quantity"]].head(3)

,date,product,price,quantity
0,10/18/2014,Cosmetics,437.20,8446
1,11/7/2011,Vegetables,154.06,3018
2,10/31/2016,Baby Food,255.28,1517


In [4]:
# Import random for generating reproducible synthetic values
import random

# Use a fixed seed so the same values are generated every time
random.seed(42)

# Possible Canadian shipping cities and coupon codes
cities = ["Toronto", "Kitchener", "Waterloo", "Ottawa", "Hamilton"]
coupons = ["NONE", "SAVE10", "SAVE15", "SAVE20"]

# Create a unique customer ID for each transaction
sales_df["customer_id"] = [
    f"CUST{i:04d}" for i in range(1, len(sales_df) + 1)
]

# Generate a coupon code for each transaction
sales_df["coupon_code"] = [
    random.choice(coupons) for _ in range(len(sales_df))
]

# Generate a shipping city for each transaction
sales_df["shipping_city"] = [
    random.choice(cities) for _ in range(len(sales_df))
]

# Display the required e-commerce fields
sales_df[
    [
        "date",
        "customer_id",
        "product",
        "price",
        "quantity",
        "coupon_code",
        "shipping_city"
    ]
].head(3)

,date,customer_id,product,price,quantity,coupon_code,shipping_city
0,10/18/2014,CUST0001,Cosmetics,437.20,8446,NONE,Toronto
1,11/7/2011,CUST0002,Vegetables,154.06,3018,NONE,Waterloo
2,10/31/2016,CUST0003,Baby Food,255.28,1517,SAVE15,Ottawa


## Step 2 - Pick the Right Container

A dictionary is useful for representing a transaction because each value can be associated with a descriptive field name such as `customer_id`, `product`, or `price`. A set is useful when only unique values are needed, such as counting unique shipping cities, while a namedtuple provides a fixed structure but is less flexible when fields need to be cleaned or transformed.

## Step 3 - Implement Functions and Data Structure

A `Transaction` class is used to represent an individual e-commerce transaction. The class stores the main transaction fields and provides methods for calculating the transaction total and cleaning individual values.

In [5]:
# Define a class for representing one e-commerce transaction
class Transaction:
    def __init__(
        self,
        date,
        customer_id,
        product,
        price,
        quantity,
        coupon_code,
        shipping_city
    ):
        self.date = date
        self.customer_id = customer_id
        self.product = product
        self.price = price
        self.quantity = quantity
        self.coupon_code = coupon_code
        self.shipping_city = shipping_city

    # Calculate the transaction total before discounts
    def total(self):
        return self.price * self.quantity

    # Clean text fields
        # Clean and standardize transaction fields
    def clean(self):
        self.product = str(self.product).strip().title()

        if pd.isna(self.coupon_code):
            self.coupon_code = "NONE"
        else:
            self.coupon_code = str(self.coupon_code).strip().upper()

        self.shipping_city = str(self.shipping_city).strip().title()

        return self

In [6]:
# Create a Transaction object from the first row of the dataset
first_transaction = Transaction(
    date=sales_df.iloc[0]["date"],
    customer_id=sales_df.iloc[0]["customer_id"],
    product=sales_df.iloc[0]["product"],
    price=sales_df.iloc[0]["price"],
    quantity=sales_df.iloc[0]["quantity"],
    coupon_code=sales_df.iloc[0]["coupon_code"],
    shipping_city=sales_df.iloc[0]["shipping_city"]
)

# Use the total() method
print("Customer:", first_transaction.customer_id)
print("Product:", first_transaction.product)
print("Transaction total:", first_transaction.total())

Customer: CUST0001
Product: Cosmetics
Transaction total: 3692591.1999999997


## Step 4 - Bulk Loaded

The DataFrame is converted into a collection of `Transaction` objects. Each row becomes one object, allowing the transaction data and its related methods to be managed together.

In [7]:
# Convert every DataFrame row into a Transaction object
transactions = [
    Transaction(
        date=row["date"],
        customer_id=row["customer_id"],
        product=row["product"],
        price=row["price"],
        quantity=row["quantity"],
        coupon_code=row["coupon_code"],
        shipping_city=row["shipping_city"]
    )
    for _, row in sales_df.iterrows()
]

# Confirm how many Transaction objects were created
print("Number of transactions:", len(transactions))

# Display the total for the first transaction
print(f"First transaction total: ${transactions[0].total():,.2f}")

Number of transactions: 500
First transaction total: $3,692,591.20


## Step 5 - Quick Profiling

Basic profiling is performed to understand the range of product prices and the number of unique shipping cities. A Python `set` is used for the city calculation because sets contain only unique values.

In [8]:
# Calculate basic price statistics
minimum_price = sales_df["price"].min()
average_price = sales_df["price"].mean()
maximum_price = sales_df["price"].max()

# Use a set to identify unique shipping cities
unique_cities = set(sales_df["shipping_city"])

# Display the profiling results
print(f"Minimum price: ${minimum_price:,.2f}")
print(f"Average price: ${average_price:,.2f}")
print(f"Maximum price: ${maximum_price:,.2f}")
print("Unique shipping cities:", len(unique_cities))
print("Cities:", unique_cities)

Minimum price: $9.33
Average price: $274.30
Maximum price: $668.27
Unique shipping cities: 5
Cities: {'Kitchener', 'Waterloo', 'Hamilton', 'Toronto', 'Ottawa'}


## Step 6 - Spot the Grime

The original source data contains no missing values or duplicate rows. To demonstrate the required data-cleaning workflow, controlled data-quality issues are introduced into selected e-commerce fields. These include missing coupon codes, inconsistent capitalization, and extra whitespace.

In [9]:
# Introduce controlled dirty-data examples for cleaning practice
sales_df.loc[0:4, "coupon_code"] = None
sales_df.loc[5:9, "shipping_city"] = " toronto "
sales_df.loc[10:14, "product"] = sales_df.loc[10:14, "product"].str.lower()

# Identify the introduced data-quality issues
missing_coupons = sales_df["coupon_code"].isna().sum()
city_whitespace = sales_df["shipping_city"].str.strip().ne(
    sales_df["shipping_city"]
).sum()
lowercase_products = sales_df["product"].apply(
    lambda value: value == value.lower()
).sum()

print("Missing coupon codes:", missing_coupons)
print("Cities with extra whitespace:", city_whitespace)
print("Lowercase product names:", lowercase_products)

Missing coupon codes: 5
Cities with extra whitespace: 5
Lowercase product names: 5


In [10]:
# Rebuild Transaction objects using the data with the introduced quality issues
transactions = [
    Transaction(
        date=row["date"],
        customer_id=row["customer_id"],
        product=row["product"],
        price=row["price"],
        quantity=row["quantity"],
        coupon_code=row["coupon_code"],
        shipping_city=row["shipping_city"]
    )
    for _, row in sales_df.iterrows()
]

print("Transaction objects rebuilt:", len(transactions))

Transaction objects rebuilt: 500


## Step 7 - Cleaning Rules

The `clean()` method of the `Transaction` class is applied to each transaction. The cleaning rules replace missing coupon codes, standardize product capitalization, and remove extra whitespace while standardizing shipping-city capitalization. Before-and-after counts are compared to verify the cleaning process.

In [11]:
# Count data-quality problems before cleaning
before_missing_coupons = sum(
    pd.isna(t.coupon_code) for t in transactions
)

before_city_whitespace = sum(
    t.shipping_city != t.shipping_city.strip()
    for t in transactions
)

before_nonstandard_products = sum(
    t.product != t.product.strip().title()
    for t in transactions
)

# Apply the clean() method to every Transaction object
for transaction in transactions:
    transaction.clean()

# Count the same problems after cleaning
after_missing_coupons = sum(
    pd.isna(t.coupon_code) for t in transactions
)

after_city_whitespace = sum(
    t.shipping_city != t.shipping_city.strip()
    for t in transactions
)

after_nonstandard_products = sum(
    t.product != t.product.strip().title()
    for t in transactions
)

# Display before-and-after results
print(
    "Missing coupon codes:",
    before_missing_coupons,
    "->",
    after_missing_coupons
)

print(
    "Cities with extra whitespace:",
    before_city_whitespace,
    "->",
    after_city_whitespace
)

print(
    "Non-standard product names:",
    before_nonstandard_products,
    "->",
    after_nonstandard_products
)

Missing coupon codes: 5 -> 0
Cities with extra whitespace: 5 -> 0
Non-standard product names: 5 -> 0


In [12]:
# Convert the cleaned Transaction objects back into a DataFrame
cleaned_df = pd.DataFrame([
    {
        "date": transaction.date,
        "customer_id": transaction.customer_id,
        "product": transaction.product,
        "price": transaction.price,
        "quantity": transaction.quantity,
        "coupon_code": transaction.coupon_code,
        "shipping_city": transaction.shipping_city
    }
    for transaction in transactions
])

# Preview the cleaned dataset
cleaned_df.head(3)

,date,customer_id,product,price,quantity,coupon_code,shipping_city
0,10/18/2014,CUST0001,Cosmetics,437.20,8446,NONE,Toronto
1,11/7/2011,CUST0002,Vegetables,154.06,3018,NONE,Waterloo
2,10/31/2016,CUST0003,Baby Food,255.28,1517,NONE,Ottawa


## Step 8 - Transformations

Coupon codes are transformed into numeric discount rates so they can be used in calculations. The discount rate is then used to calculate the final revenue for each transaction after the coupon discount is applied.

In [13]:
# Map each coupon code to a numeric discount rate
discount_map = {
    "NONE": 0.00,
    "SAVE10": 0.10,
    "SAVE15": 0.15,
    "SAVE20": 0.20
}

cleaned_df["discount_rate"] = cleaned_df["coupon_code"].map(discount_map)

# Calculate revenue before and after the coupon discount
cleaned_df["gross_revenue"] = (
    cleaned_df["price"] * cleaned_df["quantity"]
)

cleaned_df["final_revenue"] = (
    cleaned_df["gross_revenue"] * (1 - cleaned_df["discount_rate"])
)

# Preview the transformation
cleaned_df[
    [
        "coupon_code",
        "discount_rate",
        "gross_revenue",
        "final_revenue"
    ]
].head()

,coupon_code,discount_rate,gross_revenue,final_revenue
0,NONE,0.0,3692591.20,3692591.20
1,NONE,0.0,464953.08,464953.08
2,NONE,0.0,387259.76,387259.76
3,NONE,0.0,683335.40,683335.40
4,NONE,0.0,91853.85,91853.85


## Step 9 - Feature Engineering

The transaction date is converted into a datetime value and used to create a new feature called `days_since_purchase`. A fixed reference date is used instead of the current date so that the notebook produces the same results every time it is executed.

In [14]:
# Convert the transaction date from text to a datetime value
cleaned_df["date"] = pd.to_datetime(cleaned_df["date"])

# Use a fixed reference date for reproducibility
reference_date = pd.Timestamp("2026-01-01")

# Calculate the number of days between each purchase and the reference date
cleaned_df["days_since_purchase"] = (
    reference_date - cleaned_df["date"]
).dt.days

# Preview the engineered feature
cleaned_df[
    ["date", "customer_id", "days_since_purchase"]
].head()

,date,customer_id,days_since_purchase
0,2014-10-18,CUST0001,4093
1,2011-11-07,CUST0002,5169
2,2016-10-31,CUST0003,3349
3,2010-04-10,CUST0004,5745
4,2011-08-16,CUST0005,5252


## Step 10 - Mini-Aggregation

The cleaned transaction data is grouped by shipping city to calculate the total final revenue associated with each city. This demonstrates how individual transaction records can be summarized into useful business-level information.

In [15]:
# Calculate total final revenue for each shipping city
revenue_by_city = (
    cleaned_df.groupby("shipping_city")["final_revenue"]
    .sum()
    .sort_values(ascending=False)
)

# Display the aggregated results
print(revenue_by_city)

shipping_city
Ottawa       1.457866e+08
Hamilton     1.456498e+08
Waterloo     1.297746e+08
Toronto      1.070638e+08
Kitchener    1.069994e+08
Name: final_revenue, dtype: float64


## Step 11 - Serialization Checkpoint

The cleaned and transformed dataset is serialized to both CSV and JSON formats. Saving the processed data in multiple formats makes it reusable by different tools and applications.

In [16]:
# Save the cleaned dataset as CSV
cleaned_df.to_csv(
    "data/cleaned_sales_data.csv",
    index=False
)

# Save the cleaned dataset as JSON
cleaned_df.to_json(
    "data/cleaned_sales_data.json",
    orient="records",
    indent=4,
    date_format="iso"
)

print("Saved cleaned_sales_data.csv")
print("Saved cleaned_sales_data.json")

Saved cleaned_sales_data.csv
Saved cleaned_sales_data.json


## Step 12 - Soft Interview Reflection

Functions helped make this data-processing workflow more organized and reusable by grouping specific tasks into clearly defined operations. In this project, the `Transaction` class uses methods such as `total()` and `clean()` so that each transaction can calculate its own total and apply the same cleaning rules consistently. This reduces repeated code and makes the workflow easier to understand and maintain. If the cleaning rules change later, they can be updated in one place instead of changing the logic for every transaction individually.

## Secondary Metadata Source

A second open-data source containing country names and two-letter country codes is used as supporting metadata. This dataset contains 249 country and territory records with the fields `Name` and `Code`. It will be used to enrich the original sales data and contribute metadata to the final Data Dictionary.

In [17]:
# Load the secondary country metadata dataset
# keep_default_na=False preserves valid country codes such as "NA" for Namibia, otherwise we will run into an issue with Pandas interpreting
# NA as NaN later on
country_df = pd.read_csv(
    "data/country_data.csv",
    keep_default_na=False
)

# Display the dataset size and first five records
print("Country metadata rows:", len(country_df))
country_df.head()

Country metadata rows: 249


,Name,Code
0,Afghanistan,AF
1,Albania,AL
2,Algeria,DZ
3,American Samoa,AS
4,Andorra,AD


In [18]:
# Remove previous merge results if this cell is rerun
columns_to_remove = [
    "country",
    "country_code",
    "country_code_x",
    "country_code_y",
    "Name",
    "Name_clean",
    "country_match"
]

cleaned_df = cleaned_df.drop(
    columns=[
        col for col in columns_to_remove
        if col in cleaned_df.columns
    ],
    errors="ignore"
)

# Restore and clean the original country field
cleaned_df["country"] = (
    df["Country"]
    .astype(str)
    .str.replace("\u00a0", " ", regex=False)
    .str.strip()
)

# Create cleaned country names in the metadata dataset
country_df["Name_clean"] = (
    country_df["Name"]
    .astype(str)
    .str.replace("\u00a0", " ", regex=False)
    .str.strip()
)

# Map country names from the sales dataset to the exact names
# used by the secondary country metadata source
country_name_map = {
    "Central African Republic": "Central African Republic (the)",
    "Comoros": "Comoros (the)",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Democratic Republic of the Congo": "Congo (the Democratic Republic of the)",
    "Dominican Republic": "Dominican Republic (the)",
    "Federated States of Micronesia": "Micronesia (Federated States of)",
    "Iran": "Iran (Islamic Republic of)",
    "Laos": "Lao People's Democratic Republic (the)",
    "Marshall Islands": "Marshall Islands (the)",
    "Moldova": "Moldova (the Republic of)",
    "Niger": "Niger (the)",
    "North Korea": "Korea (the Democratic People's Republic of)",
    "Philippines": "Philippines (the)",
    "Republic of the Congo": "Congo (the)",
    "Russia": "Russian Federation (the)",
    "South Korea": "Korea (the Republic of)",
    "Sudan": "Sudan (the)",
    "Syria": "Syrian Arab Republic (the)",
    "Taiwan": "Taiwan (Province of China)",
    "Tanzania": "Tanzania, the United Republic of",
    "The Bahamas": "Bahamas (The)",
    "United Arab Emirates": "United Arab Emirates (the)",
    "United Kingdom": "United Kingdom of Great Britain and Northern Ireland (the)",
    "United States of America": "United States of America (the)",
    "Vatican City": "Holy See (the)",
    "Brunei": "Brunei Darussalam",
    "Cape Verde": "Cabo Verde",
    "Czech Republic": "Czechia",
    "East Timor": "Timor-Leste",
    "Macedonia": "North Macedonia",
    "Swaziland": "Eswatini",
    "Turkey": "Türkiye",
    "Vietnam": "Viet Nam"
}

# Create standardized country names for matching
cleaned_df["country_match"] = (
    cleaned_df["country"]
    .replace(country_name_map)
)

# Merge the country-code metadata into the sales data
cleaned_df = cleaned_df.merge(
    country_df[["Name_clean", "Code"]],
    how="left",
    left_on="country_match",
    right_on="Name_clean"
)

# Rename the metadata code field
cleaned_df = cleaned_df.rename(
    columns={"Code": "country_code"}
)

# Remove temporary matching columns
cleaned_df = cleaned_df.drop(
    columns=["country_match", "Name_clean"]
)

# Preview the merged data
cleaned_df[
    ["country", "country_code", "product", "final_revenue"]
].head()

,country,country_code,product,final_revenue
0,Libya,LY,Cosmetics,3692591.20
1,Canada,CA,Vegetables,464953.08
2,Libya,LY,Baby Food,387259.76
3,Japan,JP,Cereal,683335.40
4,Chad,TD,Fruits,91853.85


In [19]:
# Check how many transactions have a missing country code
missing_country_codes = cleaned_df["country_code"].isna().sum()

print(
    "Transactions with missing country codes:",
    missing_country_codes
)

# Identify countries associated with missing metadata
if missing_country_codes > 0:
    missing_code_countries = (
        cleaned_df.loc[
            cleaned_df["country_code"].isna(),
            "country"
        ]
        .drop_duplicates()
        .sort_values()
        .to_list()
    )

    print(
        "Countries with missing metadata codes:",
        missing_code_countries
    )

    print(
        "Note: Namibia has a missing Code value "
        "in the secondary metadata source."
    )
else:
    print("All transactions contain country codes.")

Transactions with missing country codes: 0
All transactions contain country codes.


## Data Dictionary

The following data dictionary documents fields from the primary sales dataset, the secondary country metadata source, and the synthetic or derived fields created during this project. The `Source` column identifies where each field originated or how it was produced.

| Field | Type | Description | Source |
|---|---|---|---|
| Region | String | Geographic sales region for the transaction. | Primary sales dataset |
| Country | String | Country associated with the original sales transaction. | Primary sales dataset |
| Item Type | String | Product category sold in the transaction; renamed to `product` in the working dataset. | Primary sales dataset |
| Sales Channel | String | Sales channel used for the transaction. | Primary sales dataset |
| Order Priority | String | Priority category assigned to the order. | Primary sales dataset |
| Order Date | Date | Date on which the order was placed; renamed to `date`. | Primary sales dataset |
| Order ID | Integer | Identifier assigned to the original order. | Primary sales dataset |
| Ship Date | Date | Date on which the order was shipped. | Primary sales dataset |
| Units Sold | Integer | Number of units sold; renamed to `quantity`. | Primary sales dataset |
| Unit Price | Float | Price per unit; renamed to `price`. | Primary sales dataset |
| Unit Cost | Float | Cost per unit. | Primary sales dataset |
| Total Revenue | Float | Original total revenue recorded for the order. | Primary sales dataset |
| Total Cost | Float | Original total cost recorded for the order. | Primary sales dataset |
| Total Profit | Float | Original total profit recorded for the order. | Primary sales dataset |
| customer_id | String | Sequential customer identifier generated for each transaction. | Synthetic |
| coupon_code | String | Coupon code generated using a fixed random seed for reproducibility. | Synthetic |
| shipping_city | String | Canadian shipping city generated using a fixed random seed for the cleaning exercise. | Synthetic |
| discount_rate | Float | Numeric discount derived from `coupon_code`. | Derived |
| gross_revenue | Float | Revenue calculated as `price × quantity` before the coupon discount. | Derived |
| final_revenue | Float | Revenue calculated after applying `discount_rate` to `gross_revenue`. | Derived |
| days_since_purchase | Integer | Number of days between the transaction date and the fixed reference date of January 1, 2026. | Derived |
| Name | String | Country or territory name used for metadata matching. | Secondary country metadata |
| country_code | String | Two-letter country code obtained from the secondary metadata source. | Secondary country metadata |

In [20]:
# Save the final enriched dataset as CSV
cleaned_df.to_csv(
    "data/cleaned_sales_data.csv",
    index=False
)

# Save the final enriched dataset as JSON
cleaned_df.to_json(
    "data/cleaned_sales_data.json",
    orient="records",
    indent=4,
    date_format="iso"
)

print("Final enriched CSV and JSON files saved successfully.")

Final enriched CSV and JSON files saved successfully.
